# 문제 2-3 : 학생 정보 구조화 시스템

자유 형식 자기소개에서 이름, 나이, 전공, 취미 리스트, 목표를 추출합니다.

- `PydanticOutputParser` 와 `BaseModel` 사용
- 각 필드에 적절한 타입과 설명(`Field(description=...)`) 추가

### 공통 설정: API Key 로드 + LLM 생성 함수

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List
from pprint import pprint


# 추출할 구조를 Pydantic 모델로 정의 (타입 + 설명)
class StudentInfo(BaseModel):
    name: str = Field(description="학생의 이름")
    age: int = Field(description="학생의 나이 (숫자만)")
    major: str = Field(description="학생의 전공")
    hobbies: List[str] = Field(description="학생의 취미 목록 (각 취미를 개별 항목으로)")
    goal: str = Field(description="학생의 목표 또는 꿈")


# Pydantic 출력 파서 생성
parser = PydanticOutputParser(pydantic_object=StudentInfo)

# 출력 형식 지침 확인 (JSON 스키마가 포함됨)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "학생의 이름", "title": "Name", "type": "string"}, "age": {"description": "학생의 나이 (숫자만)", "title": "Age", "type": "integer"}, "major": {"description": "학생의 전공", "title": "Major", "type": "string"}, "hobbies": {"description": "학생의 취미 목록 (각 취미를 개별 항목으로)", "items": {"type": "string"}, "title": "Hobbies", "type": "array"}, "goal": {"description": "학생의 목표 또는 꿈", "title": "Goal", "type": "string"}}, "required": ["name", "age", "major", "hobbies", "goal"]}
```


In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 자기소개 글에서 학생 정보를 정확히 추출하는 전문가입니다. "
               "글에 나온 정보만 사용하고, 값은 한국어로 작성하세요.\n{format_instructions}"),
    ("user", "자기소개: {intro}"),
]).partial(format_instructions=parser.get_format_instructions())

llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → Pydantic 파서
chain = prompt | llm | parser

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [5]:
intro = "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."

student = chain.invoke({"intro": intro})
print(type(student))  # StudentInfo 객체

# 필드별 접근
print(f"이름: {student.name}")
print(f"나이: {student.age} (타입: {type(student.age).__name__})")
print(f"전공: {student.major}")
print(f"취미: {student.hobbies}")
print(f"목표: {student.goal}")

# dict / JSON 형태로 출력
print(student.model_dump_json(indent=4))

<class '__main__.StudentInfo'>
이름: 김민수
나이: 22 (타입: int)
전공: 컴퓨터공학
취미: ['게임하기', '영화보기', '코딩']
목표: 훌륭한 개발자가 되는 것
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "훌륭한 개발자가 되는 것"
}


In [6]:
from langchain_core.exceptions import OutputParserException

# 다른 자기소개로도 테스트
intros = [
    "반갑습니다~ 경영학과 3학년 이서연이에요. 올해 스물네 살이고요, 주말엔 등산이랑 사진 찍기를 즐겨요. 졸업 후에는 마케팅 전문가로 일하고 싶어요.",
    "제 이름은 박지훈, 나이는 20살! 전공은 시각디자인입니다. 그림 그리기, 피아노 연주, 고양이 돌보기가 취미예요. 언젠가 제 이름을 건 디자인 스튜디오를 여는 게 꿈입니다.",
]

for text in intros:
    try:
        pprint(chain.invoke({"intro": text}).model_dump())
    except OutputParserException as e:
        print(f"파싱 오류: {str(e).splitlines()[0]}")
    print("-" * 50)

{'age': 24,
 'goal': '마케팅 전문가가 되는 것',
 'hobbies': ['등산', '사진 찍기'],
 'major': '경영학',
 'name': '이서연'}
--------------------------------------------------
{'age': 20,
 'goal': '제 이름을 건 디자인 스튜디오를 여는 것이 꿈입니다.',
 'hobbies': ['그림 그리기', '피아노 연주', '고양이 돌보기'],
 'major': '시각디자인',
 'name': '박지훈'}
--------------------------------------------------
